<div align="center">

# Patra Toolkit: Model Card & Datasheet Demo

</div>

This notebook is a focused, end-to-end walkthrough of the core Patra Toolkit operations against the Postgres-backed Patra server:

1. Model Card creation
2. Datasheet creation
3. Model Card submission
4. Datasheet submission
5. Model Card listing
6. Datasheet listing
7. Model Card retrieval
8. Datasheet retrieval

It uses placeholder metadata rather than training a real model, so it can be run end-to-end quickly. For a fuller example that trains a real model and runs fairness/explainability scanners, see `GettingStarted.ipynb`.

In [ ]:
!pip install patra-toolkit

In [ ]:
from patra_toolkit import ModelCard, AIModel, Datasheet, run_experiment
from patra_toolkit.datasheet import DatasheetAlternateIdentifier

## Configuration

Set the base URL of your Patra server. If you're running the backend locally via `docker compose -f docker-compose.backend.yml up --build` in `patra-knowledge-base`, this defaults to `http://localhost:8000`.

In [ ]:
patra_server_url = "http://localhost:8000"

## 1. Model Card Creation

Only `name` is required on `ModelCard` and `AIModel` -- every other field is optional. Attach an `AIModel` to describe the underlying model.

In [ ]:
mc = ModelCard(
    name="Demo_Classification_Model",
    version="1.0",
    short_description="A placeholder model card for demonstrating the Patra Toolkit API.",
    full_description=(
        "This model card does not describe a real trained model -- it exists purely to demonstrate "
        "model card creation, submission, listing, and retrieval."
    ),
    keywords="demo, patra, toolkit",
    author="Demo Author",
    input_type="Tabular",
    category="classification",
)

ai_model = AIModel(
    name="DemoModel",
    version="1.0",
    description="Placeholder AI model metadata.",
    owner="Demo Author",
    framework="sklearn",
    model_type="random_forest",
    test_accuracy=0.87,
)
ai_model.add_metric("Precision", 0.85)
ai_model.add_metric("Recall", 0.83)

mc.ai_model = ai_model
mc.validate()

## 2. Datasheet Creation

Datasheets describe datasets using DataCite-style metadata. No fields are required to construct one, but `validate()`/`submit()` require at least one title and one creator.

In [ ]:
ds = Datasheet(publication_year=2025, version="1.0")
ds.add_title("Demo Dataset")
ds.add_creator("Demo Author")
ds.add_description("A placeholder dataset for demonstrating the Patra Toolkit API.", "Abstract")

ds.validate()

## [Optional] TAPIS Authentication

Patra servers hosted as TAPIS pods require authentication using a JWT for secure access. To generate this token, authenticate with your TACC credentials. If you do not already have a TACC account, you can create one at [https://accounts.tacc.utexas.edu/begin](https://accounts.tacc.utexas.edu/begin). If your Patra server doesn't require authentication, skip this cell and pass `token=None` when submitting.

In [ ]:
tapis_token = mc.authenticate(username="<your_tacc_username>", password="<your_tacc_password>")
# tapis_token = None  # uncomment if your Patra server doesn't require authentication

## 3. Model Card Submission

In [ ]:
mc_result = mc.submit(patra_server_url=patra_server_url, token=tapis_token)
print(mc_result)
print("Model Card uuid:", mc.uuid)

## 4. Datasheet Submission

In [ ]:
ds_result = ds.submit(patra_server_url=patra_server_url, token=tapis_token)
print(ds_result)
print("Datasheet uuid:", ds.uuid)

## 5. Model Card Listing

Returns summaries (`uuid`, `name`, and other summary fields) for model cards on the server.

In [ ]:
ModelCard.list_model_cards(server_url=patra_server_url, token=tapis_token, q="Demo_Classification_Model")

## 6. Datasheet Listing

Returns summaries (`uuid`, `title`, and other summary fields) for datasheets on the server.

In [ ]:
Datasheet.list_datasheets(server_url=patra_server_url, token=tapis_token, q="Demo Dataset")

## 7. Model Card Retrieval

Retrieves the full record for a single model card by `uuid`, including its nested `ai_model` details.

In [ ]:
ModelCard.get_model_card(server_url=patra_server_url, uuid=mc.uuid, token=tapis_token)

## 8. Datasheet Retrieval

Retrieves the full DataCite-style record for a single datasheet by `uuid`.

In [ ]:
Datasheet.get_datasheet(server_url=patra_server_url, uuid=ds.uuid, token=tapis_token)

## Summary

By following this notebook, you have:
1. Created a Model Card and a Datasheet
2. [Optionally] Authenticated with TAPIS to obtain a token
3. Submitted both the Model Card and the Datasheet to a Patra server
4. Listed model cards and datasheets on the server
5. Retrieved a single model card and a single datasheet by `uuid`

## 9. Inference Experiment + Streaming to CKN

This section runs a small real inference experiment -- get a Model Card and Datasheet from Patra, then hand their uuids to `run_experiment()`, which downloads the referenced model and sample images, runs inference, and streams per-image metrics to CKN (`cyberinfrastructure-knowledge-network`) as it goes. Those events flow through CKN's existing Kafka Connect sink into Patra's Postgres `events` table, where they're visible in the Patra frontend's **Digital Agriculture** experiments page.

`run_experiment()` connects directly to a real, already-running CKN Kafka broker -- there's no local infrastructure to stand up for this section.

### Prerequisites

1. A reachable CKN Kafka broker address (e.g. `cknbroker.pods.icicleai.tapis.io:443` -- use whatever your broker's *advertised* external listener actually is, which may not match the port you'd otherwise expect).
2. A `user_id` that's already registered in the `users` table wherever these events land -- `run_experiment()` does not auto-register users, and there's no safe default.
3. `pip install "patra-toolkit[experiments]"` (installs torch, torchvision, pillow, and confluent-kafka).

If your CKN deployment's Kafka Connect sink connector doesn't require the schema-enveloped JSON format (some do, some accept bare JSON), pass `use_schema_envelope=False` to `run_experiment()`.

In [ ]:
!pip install "patra-toolkit[experiments]"

In [ ]:
# torchvision is needed here (outside run_experiment()) just to read the pretrained
# weights' real download URL and category list for the Model Card's metadata below.
import torchvision

### 9.1 Build and submit a Model Card + Datasheet for the inference model

These are new, separate objects from the `mc`/`ds` used earlier in this notebook -- keeping them distinct means re-running the notebook top-to-bottom stays coherent, since the earlier list/retrieval cells describe the placeholder sklearn card, not this one.

`ai_model.location` is set to torchvision's real, publicly downloadable MobileNetV2 weights URL -- reading `weights.url` here doesn't download anything; `run_experiment()` does the actual download.

In [ ]:
weights = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1
weights_url = weights.url
imagenet_categories = weights.meta["categories"]
top1_acc = weights.meta["_metrics"]["ImageNet-1K"]["acc@1"] / 100.0

inference_ai_model = AIModel(
    name="MobileNetV2_ImageNet",
    version="1.0",
    description="Torchvision MobileNetV2 CNN pretrained on ImageNet-1k; used for a live inference-streaming demo.",
    owner="Demo Author",
    location=weights_url,
    license="BSD-3-Clause",
    framework="pytorch",
    model_type="cnn",
    test_accuracy=round(top1_acc, 5),
    inference_labels=imagenet_categories,
)

inference_mc = ModelCard(
    name="MobileNetV2_Inference_Demo",
    version="1.0",
    short_description="Real pretrained MobileNetV2 used to demonstrate inference + CKN streaming.",
    full_description=(
        "Downloads a real ImageNet-pretrained MobileNetV2 checkpoint via ai_model.location, runs it "
        "over sample images, and streams per-image inference metrics to CKN."
    ),
    keywords="demo, patra, ckn, inference, mobilenetv2",
    author="Demo Author",
    input_type="Image",
    category="classification",
)
inference_mc.ai_model = inference_ai_model
inference_mc.validate()

inference_ds = Datasheet(publication_year=2026, version="1.0")
inference_ds.add_title("CKN Inference Demo Images")
inference_ds.add_creator("Demo Author")
inference_ds.alternate_identifiers.append(
    DatasheetAlternateIdentifier(alternate_identifier="https://picsum.photos", alternate_identifier_type="URL")
)
inference_ds.add_description(
    "Images fetched from Lorem Picsum via https://picsum.photos/id/{n}/224/224 for a live "
    "inference-streaming demo. These are arbitrary real-world photographs with no ImageNet ground-truth labels.",
    "TechnicalInfo",
)
inference_ds.validate()

In [ ]:
inference_mc_result = inference_mc.submit(patra_server_url=patra_server_url, token=tapis_token)
inference_ds_result = inference_ds.submit(patra_server_url=patra_server_url, token=tapis_token)
print("Model Card uuid:", inference_mc.uuid)
print("Datasheet uuid:", inference_ds.uuid)

### 9.2 Run the experiment

`run_experiment()` does the rest on its own: downloads the Model Card and Datasheet by uuid, downloads the model weights and sample images they reference, runs inference, and streams a CKN event per image as it's processed.

In [ ]:
result = run_experiment(
    model_card_uuid=inference_mc.uuid,
    datasheet_uuid=inference_ds.uuid,
    patra_server_url=patra_server_url,
    ckn_broker_url="cknbroker.pods.icicleai.tapis.io:443",  # use your CKN broker's advertised external address
    user_id="<your_registered_user_id>",
    token=tapis_token,
)
result

### 9.3 View results in the Patra frontend

1. Backend: `ENABLE_DOMAIN_EXPERIMENTS` needs to be enabled on the Patra server this experiment was streamed against.
2. Frontend: set `VITE_SUPPORTS_DOMAIN_EXPERIMENTS=true` in `patra-frontend/app/.env` (it defaults to `false`), then `npm run dev` from `patra-frontend/app/`.
3. Open the app and click **Digital Agriculture** under Experiments in the sidebar, then select your `user_id` to see this run's summary and per-image results.

You can also check the same data the frontend reads directly via the REST API -- `result['results_url']` above is exactly that endpoint:
```bash
curl -s "$(python3 -c "print(result['results_url'])")"
```

**If nothing shows up**, check your CKN deployment's Kafka Connect logs for errors around the time you ran this -- the sink connector silently drops malformed or unresolvable records (e.g. an unregistered `user_id` or `model_id`, or a schema-envelope mismatch -- see `use_schema_envelope` above) rather than raising anything visible here.